<a href="https://colab.research.google.com/github/IMPERIUMSIXTH/JupyterNotebooks/blob/main/SmartPhone_Addiction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
!pip install -q polars xgboost lightgbm imbalanced-learn

import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Fast ML libraries
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from imblearn.over_sampling import SMOTE
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

import joblib
import gc

np.random.seed(42)
tf.random.set_seed(42)

print("✅ All libraries imported!")
print(f"TensorFlow: {tf.__version__}")

✅ All libraries imported!
TensorFlow: 2.20.0


In [13]:
from google.colab import files
uploaded=files.upload()

# Get the name of the uploaded file
uploaded_filename = list(uploaded.keys())[0]

# Load with Polars (already fast)
train_df = pl.read_csv(uploaded_filename)
print(f"✅ Loaded {train_df.shape[0]:,} rows, {train_df.shape[1]} columns")

Saving train.csv to train (2).csv
✅ Loaded 691,369 rows, 14 columns


In [16]:
# Re-load train_df from the uploaded file to ensure it's available
# This handles cases where the kernel might have reset or variables were cleared.
if 'uploaded_filename' in globals():
    train_df = pl.read_csv(globals()['uploaded_filename'])
    print(f"train_df re-loaded from {globals()['uploaded_filename']}")
else:
    raise NameError("uploaded_filename not found. Please ensure the data loading cell (cWMgN528XZz7) has been executed.")

# Convert to pandas (needed for sklearn)
df = train_df.to_pandas()

# Quick check
print(f"✅ Working with {len(df):,} rows")

# Separate features and target
if 'id' in df.columns:
    df = df.drop('id', axis=1)
X = df.drop('addicted_label', axis=1)
y = df['addicted_label']

print(f"✅ Features: {X.shape[1]}, Target: {y.shape[0]}")

# ============================================================================
# STEP 1: FAST CATEGORICAL ENCODING (pandas factorize)
# ============================================================================

print("\n🔤 Encoding categoricals...")
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
for col in categorical_cols:
    X[col] = X[col].fillna('Unknown').astype(str)
    X[col] = pd.factorize(X[col])[0]  # Fastest method

print(f"✅ Encoded {len(categorical_cols)} columns")

# ============================================================================
# STEP 2: FAST IMPUTATION (SimpleImputer)
# ============================================================================

print("\n🔢 Imputing missing values...")
numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()

# Use SimpleImputer (fast)
imputer = SimpleImputer(strategy='median')
X_imputed = imputer.fit_transform(X[numeric_cols])

print(f"✅ Imputed {len(numeric_cols)} numeric columns")
print(f"   Memory: {X_imputed.nbytes / 1024**2:.1f} MB")

# ============================================================================
# STEP 3: FAST SCALING (StandardScaler)
# ============================================================================

print("\n📏 Scaling features...")
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)
print(f"✅ Scaled {X_scaled.shape[1]} features")
print(f"   Memory: {X_scaled.nbytes / 1024**2:.1f} MB")

# ============================================================================
# STEP 4: FAST FEATURE ENGINEERING (vectorized)
# ============================================================================

print("\n🔧 Creating engineered features...")

# Create DataFrame from scaled data
X_engineered = pd.DataFrame(X_scaled, columns=numeric_cols)

# Vectorized operations (fast)
X_engineered['screen_time_ratio'] = (
    X_engineered['weekend_screen_time'] / (X_engineered['daily_screen_time_hours'] + 0.01)
)
X_engineered['social_media_ratio'] = (
    X_engineered['social_media_hours'] / (X_engineered['daily_screen_time_hours'] + 0.01)
)
X_engineered['gaming_ratio'] = (
    X_engineered['gaming_hours'] / (X_engineered['daily_screen_time_hours'] + 0.01)
)
X_engineered['engagement_score'] = (
    X_engineered['notifications_per_day'] * X_engineered['app_opens_per_day']
) / 100
X_engineered['entertainment_hours'] = (
    X_engineered['social_media_hours'] + X_engineered['gaming_hours']
)
X_engineered['productivity_ratio'] = (
    X_engineered['work_study_hours'] / (X_engineered['entertainment_hours'] + 0.01)
)
X_engineered['sleep_deficit'] = np.maximum(0, 8 - X_engineered['sleep_hours'])
X_engineered['total_screen'] = (
    X_engineered['daily_screen_time_hours'] + X_engineered['weekend_screen_time']
)

print(f"✅ Added 8 engineered features")
print(f"✅ Final features: {X_engineered.shape[1]}")

# ============================================================================
# FINAL PREPARATION
# ============================================================================

X_final = X_engineered.values
y_final = y.values

print(f"\n📊 Final shape: {X_final.shape[0]:,} rows, {X_final.shape[1]} features")
print(f"💾 Memory: {X_final.nbytes / 1024**2:.1f} MB")

# Save preprocessors
joblib.dump(imputer, 'imputer.pkl')
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(numeric_cols, 'feature_columns.pkl')

print("\n✅ Preprocessing complete! (< 1 minute)")
print("="*80)

# Free up memory
del X_imputed, X_scaled, X_engineered, X
gc.collect()

train_df re-loaded from train (2).csv
✅ Working with 691,369 rows
✅ Features: 12, Target: 691369

🔤 Encoding categoricals...
✅ Encoded 3 columns

🔢 Imputing missing values...
✅ Imputed 12 numeric columns
   Memory: 63.3 MB

📏 Scaling features...
✅ Scaled 12 features
   Memory: 63.3 MB

🔧 Creating engineered features...
✅ Added 8 engineered features
✅ Final features: 20

📊 Final shape: 691,369 rows, 20 features
💾 Memory: 105.5 MB

✅ Preprocessing complete! (< 1 minute)


462618

In [17]:
def create_fast_nn(input_shape):
    """Smaller, faster neural network"""
    model = keras.Sequential([
        layers.Dense(64, activation='relu', input_shape=(input_shape,)),
        layers.Dropout(0.3),
        layers.Dense(32, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy', 'auc']
    )
    return model

# Create and train
nn_model = create_fast_nn(X_final.shape[1])

# Fast training with early stopping
early_stop = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

print("🚀 Training neural network...")
history = nn_model.fit(
    X_final, y_final,
    epochs=30,  # Reduced epochs
    batch_size=2048,  # Larger batch = faster
    validation_split=0.15,
    callbacks=[early_stop],
    verbose=1
)

print(f"✅ Neural network trained in {len(history.history['loss'])} epochs")
print(f"📊 Validation AUC: {max(history.history['val_auc']):.4f}")

🚀 Training neural network...
Epoch 1/30
287/287 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - accuracy: 0.7915 - auc: 0.8437 - loss: 0.5610 - val_accuracy: 0.8361 - val_auc: 0.9112 - val_loss: 0.3661
Epoch 2/30
287/287 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.8294 - auc: 0.8998 - loss: 0.4009 - val_accuracy: 0.8400 - val_auc: 0.9171 - val_loss: 0.3508
Epoch 3/30
287/287 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.8363 - auc: 0.9095 - loss: 0.3681 - val_accuracy: 0.8414 - val_auc: 0.9198 - val_loss: 0.3297
Epoch 4/30
287/287 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.8394 - auc: 0.9139 - loss: 0.3448 - val_accuracy: 0.8440 - val_auc: 0.9224 - val_loss: 0.3208
Epoch 5/30
287/287 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.8414 - auc: 0.9168 - loss: 0.3347 - val_accuracy: 0.8467 - val_auc: 0.9240 - val_loss: 0.3138
Epoch 6/30
287/287 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.8433 - auc: 0.9188 - loss: 0.3273 - val_accuracy: 0.8494 - val_auc: 0.9253 - val_loss: 0.3099
Epoch 7/30


In [19]:
# Sample data for faster training (if needed)
sample_size = min(200000, len(X_final))  # Use subset for speed
if len(X_final) > sample_size:
    print(f"⚠️ Sampling {sample_size:,} rows for faster training...")
    indices = np.random.choice(len(X_final), sample_size, replace=False)
    X_sample = X_final[indices]
    y_sample = y_final[indices]
else:
    X_sample = X_final
    y_sample = y_final

# SMOTE on sampled data (faster)
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_sample, y_sample)
print(f"✅ SMOTE: {len(X_sample):,} → {len(X_resampled):,} samples")

# Define fast models with reduced complexity
models = {
    'RF': RandomForestClassifier(
        n_estimators=100,  # Reduced from 200
        max_depth=10,      # Reduced depth
        n_jobs=-1,
        random_state=42
    ),
    'XGB': XGBClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        n_jobs=-1,
        random_state=42,
        use_label_encoder=False,
        eval_metric='logloss'
    ),
    'LGBM': LGBMClassifier(
        n_estimators=100,
        max_depth=6,
        num_leaves=15,
        learning_rate=0.1,
        n_jobs=-1,
        random_state=42,
        verbose=-1
    )
}

trained_models = {}
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)  # Reduced folds

for name, model in models.items():
    print(f"\n📊 Training {name}...")

    # Fast CV
    cv_scores = cross_val_score(model, X_resampled, y_resampled,
                                cv=cv, scoring='roc_auc', n_jobs=-1)
    print(f"   CV AUC: {cv_scores.mean():.4f} (±{cv_scores.std():.4f})")

    # Train
    model.fit(X_resampled, y_resampled)
    trained_models[name] = model

# Voting ensemble
voting_clf = VotingClassifier(
    estimators=[(name, model) for name, model in trained_models.items()],
    voting='soft'
)
voting_clf.fit(X_resampled, y_resampled)

print("\n✅ All models trained!")

# Save models
joblib.dump(voting_clf, 'ensemble_model.pkl')

⚠️ Sampling 200,000 rows for faster training...
✅ SMOTE: 200,000 → 283,760 samples

📊 Training RF...
   CV AUC: 0.9423 (±0.0002)

📊 Training XGB...
   CV AUC: 0.9689 (±0.0002)

📊 Training LGBM...
   CV AUC: 0.9668 (±0.0006)

✅ All models trained!


['ensemble_model.pkl']

In [20]:
def fast_evaluate(model, X, y, name):
    """Fast model evaluation"""
    if hasattr(model, 'predict_proba'):
        y_proba = model.predict_proba(X)[:, 1]
    else:
        y_proba = model.predict(X)
    y_pred = (y_proba > 0.5).astype(int)

    return {
        'Model': name,
        'Accuracy': accuracy_score(y, y_pred),
        'Precision': precision_score(y, y_pred),
        'Recall': recall_score(y, y_pred),
        'F1': f1_score(y, y_pred),
        'AUC': roc_auc_score(y, y_proba)
    }

results = []

# Evaluate all models fast
print("🔍 Evaluating models...")

# Neural Network
nn_pred = nn_model.predict(X_final, verbose=0).flatten()
nn_y_pred = (nn_pred > 0.5).astype(int)
results.append({
    'Model': 'Neural Network',
    'Accuracy': accuracy_score(y_final, nn_y_pred),
    'Precision': precision_score(y_final, nn_y_pred),
    'Recall': recall_score(y_final, nn_y_pred),
    'F1': f1_score(y_final, nn_y_pred),
    'AUC': roc_auc_score(y_final, nn_pred)
})

# Ensemble
ensemble_proba = voting_clf.predict_proba(X_final)[:, 1]
ensemble_pred = (ensemble_proba > 0.5).astype(int)
results.append({
    'Model': 'Voting Ensemble',
    'Accuracy': accuracy_score(y_final, ensemble_pred),
    'Precision': precision_score(y_final, ensemble_pred),
    'Recall': recall_score(y_final, ensemble_pred),
    'F1': f1_score(y_final, ensemble_pred),
    'AUC': roc_auc_score(y_final, ensemble_proba)
})

# Individual models
for name, model in trained_models.items():
    results.append(fast_evaluate(model, X_final, y_final, name))

# Create comparison
results_df = pd.DataFrame(results)
print("\n📊 MODEL COMPARISON:")
print(results_df.round(4).to_string(index=False))

# Best model
best_idx = results_df['AUC'].idxmax()
best_model_name = results_df.loc[best_idx, 'Model']
best_auc = results_df.loc[best_idx, 'AUC']

print(f"\n🏆 Best Model: {best_model_name} (AUC: {best_auc:.4f})")

🔍 Evaluating models...

📊 MODEL COMPARISON:
          Model  Accuracy  Precision  Recall     F1    AUC
 Neural Network    0.8611     0.8911  0.9162 0.9035 0.9344
Voting Ensemble    0.8523     0.9377  0.8481 0.8907 0.9400
             RF    0.8373     0.9525  0.8112 0.8762 0.9362
            XGB    0.8601     0.9305  0.8676 0.8980 0.9415
           LGBM    0.8558     0.9301  0.8614 0.8945 0.9386

🏆 Best Model: XGB (AUC: 0.9415)


In [21]:
# Load test data
from google.colab import files
uploaded=files.upload()

# Get the name of the uploaded file
uploaded_filename = list(uploaded.keys())[0]

# Load with Polars (already fast)
test_df = pl.read_csv(uploaded_filename)

# Quick preprocessing function
def fast_preprocess_test(test_df, imputer, scaler, feature_cols):
    """Fast preprocessing for test data"""
    test_pd = test_df.to_pandas()

    # Remove ID
    if 'id' in test_pd.columns:
        ids = test_pd['id'].copy()
        test_pd = test_pd.drop('id', axis=1)
    else:
        ids = None

    # Handle categoricals
    cat_cols = test_pd.select_dtypes(include=['object']).columns.tolist()
    for col in cat_cols:
        test_pd[col] = test_pd[col].fillna('Unknown').astype(str)
        test_pd[col] = pd.factorize(test_pd[col])[0]

    # Impute
    test_imputed = imputer.transform(test_pd[feature_cols])

    # Scale
    test_scaled = scaler.transform(test_imputed)

    # Feature engineering
    test_engineered = pd.DataFrame(test_scaled, columns=feature_cols)
    test_engineered['screen_time_ratio'] = (
        test_engineered['weekend_screen_time'] / (test_engineered['daily_screen_time_hours'] + 0.01)
    )
    test_engineered['social_media_ratio'] = (
        test_engineered['social_media_hours'] / (test_engineered['daily_screen_time_hours'] + 0.01)
    )
    test_engineered['gaming_ratio'] = (
        test_engineered['gaming_hours'] / (test_engineered['daily_screen_time_hours'] + 0.01)
    )
    test_engineered['engagement_score'] = (
        test_engineered['notifications_per_day'] * test_engineered['app_opens_per_day']
    ) / 100
    test_engineered['entertainment_hours'] = (
        test_engineered['social_media_hours'] + test_engineered['gaming_hours']
    )
    test_engineered['productivity_ratio'] = (
        test_engineered['work_study_hours'] / (test_engineered['entertainment_hours'] + 0.01)
    )
    test_engineered['sleep_deficit'] = np.maximum(0, 8 - test_engineered['sleep_hours'])
    test_engineered['total_screen'] = (
        test_engineered['daily_screen_time_hours'] + test_engineered['weekend_screen_time']
    )

    return test_engineered.values, ids

# Preprocess test
X_test, test_ids = fast_preprocess_test(test_df, imputer, scaler, numeric_cols)
print(f"✅ Test data preprocessed: {X_test.shape}")

# Get best model
if best_model_name == 'Voting Ensemble':
    best_model = voting_clf
elif best_model_name == 'Neural Network':
    best_model = nn_model
else:
    best_model = trained_models[best_model_name]

# Predict
print("\n🔮 Making predictions...")
if hasattr(best_model, 'predict_proba'):
    predictions = best_model.predict_proba(X_test)[:, 1]
else:
    predictions = best_model.predict(X_test).flatten()

predictions_binary = (predictions > 0.5).astype(int)

# Create submission
submission = pd.DataFrame({
    'id': test_ids if test_ids is not None else range(len(predictions)),
    'addicted_label': predictions_binary,
    'probability': predictions
})

print(f"\n✅ Predictions complete!")
print(f"📊 Prediction distribution:")
print(pd.Series(predictions_binary).value_counts())

# Save submission
submission.to_csv('submission.csv', index=False)
print("\n✅ Submission saved as 'submission.csv'")

# Show sample
print("\n📋 Sample predictions:")
print(submission.head(10))

Saving test.csv to test.csv
✅ Test data preprocessed: (296302, 20)

🔮 Making predictions...

✅ Predictions complete!
📊 Prediction distribution:
1    196348
0     99954
Name: count, dtype: int64

✅ Submission saved as 'submission.csv'

📋 Sample predictions:
       id  addicted_label  probability
0  691369               1     0.995050
1  691370               1     0.898290
2  691371               1     0.955186
3  691372               1     0.955025
4  691373               1     0.983515
5  691374               1     0.750982
6  691375               1     0.831368
7  691376               0     0.409940
8  691377               1     0.963534
9  691378               1     0.719732


In [22]:
print(f"""
📊 DATASET:
   Training: {len(df):,} rows, {X_final.shape[1]} features
   Test: {test_df.shape[0]:,} rows

🏆 BEST MODEL: {best_model_name}
   AUC-ROC: {best_auc:.4f}
   Accuracy: {results_df.loc[best_idx, 'Accuracy']:.4f}
   F1-Score: {results_df.loc[best_idx, 'F1']:.4f}

⚡ PERFORMANCE:
   Preprocessing: < 1 minute
   Model Training: < 3 minutes
   Total Time: < 5 minutes

💾 SAVED FILES:
   • imputer.pkl
   • scaler.pkl
   • ensemble_model.pkl
   • submission.csv

🚀 SUBMISSION READY!
""")


📊 DATASET:
   Training: 691,369 rows, 20 features
   Test: 296,302 rows

🏆 BEST MODEL: XGB
   AUC-ROC: 0.9415
   Accuracy: 0.8601
   F1-Score: 0.8980

⚡ PERFORMANCE:
   Preprocessing: < 1 minute
   Model Training: < 3 minutes
   Total Time: < 5 minutes

💾 SAVED FILES:
   • imputer.pkl
   • scaler.pkl
   • ensemble_model.pkl
   • submission.csv

🚀 SUBMISSION READY!



In [23]:
# Load sample_submission.csv to inspect its format
sample_submission_df = pd.read_csv('/content/sample_submission.csv')
print("\nSample Submission File Columns:")
print(sample_submission_df.columns.tolist())

# Create a new submission DataFrame with only 'id' and 'addicted_label' to match the sample format
final_submission = submission[['id', 'addicted_label']]

# Save the final submission file
final_submission.to_csv('submission.csv', index=False)
print("\n✅ Final submission.csv created with 'id' and 'addicted_label' columns.")
print("\n📋 First 5 rows of final submission.csv:")
print(final_submission.head())


Sample Submission File Columns:
['id', 'addicted_label']

✅ Final submission.csv created with 'id' and 'addicted_label' columns.

📋 First 5 rows of final submission.csv:
       id  addicted_label
0  691369               1
1  691370               1
2  691371               1
3  691372               1
4  691373               1
